# 工具
工具 是 agents 调用来执行操作的组件。

## 1、工具调用


## 创建工具
创建工具最简单的方法是使用 @tool 装饰器。

In [ ]:

from langchain.tools import tool

# 定义工具
@tool
def get_weather(city:str) -> str:
    """
    获取天气信息
    """
    return city + "天气晴朗，气温 20摄氏度"

# 直接调用
get_weather.invoke({"city":"北京"})

'北京天气晴朗，气温 20摄氏度'

In [11]:
from langchain_ollama import ChatOllama
from langchain.tools import tool

# 定义工具
@tool
def get_weather(city:str) -> str:
    """
    获取指定城市天气信息
    """
    return city + "天气晴朗，气温 20摄氏度"

# 提供llm
model = ChatOllama(
    model="deepseek-r1:14b",
)

# 绑定工具
model_with_tools = model.bind_tools([get_weather])

res = model_with_tools.invoke("上海天气如何？")

if res.tool_calls:
    print("ai 想调用工具：", res.tool_calls)
else:
    print("ai 不想调用工具：", res.content)

ai 不想调用工具： 

您好，建议您联网获取时效性较强的信息；如果还有其他问题需要帮助，请随时告诉我！


In [13]:
import os
from langchain.chat_models import init_chat_model
from langchain.tools import tool

from dotenv import load_dotenv
# 需要显示加载配置文件
load_dotenv()  # 默认读取当前目录的 .env

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 初始化模型
model = init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)

# 定义工具
@tool
def get_weather(city:str) -> str:
    """
    获取指定城市天气信息
    """
    return city + "天气晴朗，气温 20摄氏度"


# 绑定工具
model_with_tools = model.bind_tools([get_weather])

res = model_with_tools.invoke("上海天气如何？")

if res.tool_calls:
    print("ai 想调用工具：", res.tool_calls)
else:
    print("ai 不想调用工具：", res.content)
    
model_with_tools.invoke("调用本地ollama部署大模型不会调用工单为什么？")

ai 想调用工具： [{'name': 'get_weather', 'args': {'city': '上海'}, 'id': 'call_00_I4BO0EwhxpTUMg7trxH24229', 'type': 'tool_call'}]


AIMessage(content='# 本地 Ollama 部署的模型不调用工具（工单/Function Calling）的常见原因\n\n这通常不是"玄学"，而是下面几个环节之一出了问题。按可能性从高到低排查：\n\n## 1. 模型本身不支持 Tool Calling（最常见）\n不是所有模型都经过工具调用训练。**必须选明确支持 tools 的模型**，例如：\n- `llama3.1` / `llama3.2`\n- `qwen2.5`（7B 及以上）\n- `mistral-nemo`、`firefunction-v2`\n- `mistral-small` 等\n\n像 `gemma2`、`llama2`、`deepseek-coder`(部分版本) 等，即使 Ollama 支持 API，模型也不会真正产出结构化 tool_calls，只会在正文里"假装"写一段 JSON 文本。\n\n## 2. Ollama 版本太旧\n`/api/chat` 的 `tools` 参数是较新版本才支持的（大致 0.3.x 之后）。用 `ollama --version` 查看，建议升级到最新版。\n\n## 3. 请求里没带 `tools`，或格式不对\n工具要通过 API 的 `tools` 字段传入，而不是塞进 prompt 里。示例：\n\n```bash\ncurl http://localhost:11434/api/chat -d \'{\n  "model": "qwen2.5:7b",\n  "messages": [{"role":"user","content":"北京天气怎么样"}],\n  "tools": [{\n    "type": "function",\n    "function": {\n      "name": "get_weather",\n      "description": "查询城市天气",\n      "parameters": {\n        "type": "object",\n        "properties": {"city": {"type":"string"}},\n        "required": ["city"]\n      }\n    }\n  }],\n  "stre

In [23]:
import os
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage,AIMessage
from langchain.tools import tool
from rich import print as rprint

from dotenv import load_dotenv
# 需要显示加载配置文件
load_dotenv()  # 默认读取当前目录的 .env

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 初始化模型
model = init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)

# 定义工具
@tool
def get_weather(city:str) -> str:
    """
    获取指定城市天气信息
    """
    return city + "天气晴朗，气温 20摄氏度"


# 绑定工具
model_with_tools = model.bind_tools([get_weather])

messages = [
    HumanMessage("上海天气如何？")
]

# 调用
response = model_with_tools.invoke(messages)
# rprint(response) # 可以看详细响应
messages.append(response)

# 获取响应中的工具调用信息对象
tool_calls = response.tool_calls

for tool_call in tool_calls:
    if tool_call["name"] == "get_weather":
        tool_res = get_weather.invoke(tool_call)
        messages.append(tool_res)

print("#"*10 + "message" + "#"*10)
# print(messages)
for msg in messages:
    msg.pretty_print()
print("#"*10 + "message" + "#"*10)

final_response = model_with_tools.invoke(messages)
print(final_response)

##########message##########
================================ Human Message =================================

上海天气如何？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_RWzneAYZiZKOw0uilvKF9654)
 Call ID: call_00_RWzneAYZiZKOw0uilvKF9654
  Args:
    city: 上海
================================= Tool Message =================================
Name: get_weather

上海天气晴朗，气温 20摄氏度
##########message##########
content='上海目前天气晴朗，气温 20 摄氏度，是比较舒适的好天气。外出的话建议适当防晒，早晚温差可能略大，可以带一件薄外套备用。' additional_kwargs={'refusal': None, 'reasoning_content': "The tool returned weather info. I'll relay it."} response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 367, 'total_tokens': 414, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 11, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cach